# Gold Location Trends Mart

**Purpose**: Dashboard-ready geographic hiring patterns and trends.

**Target Table**: `workspace.gold.gold_location_trends`

**Grain**: One row per date per location per sector

**Key Metrics**:
- Active jobs by location
- New job postings
- Unique companies hiring in each location
- Unique roles available
- Geographic hiring concentration

**Data Sources**:
- `workspace.warehouse.fact_job_postings`
- `workspace.warehouse.dim_location`

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_location_trends
USING DELTA
COMMENT 'Geographic hiring trends for dashboard consumption'
AS

WITH base_metrics AS (
  SELECT
    f.posting_date_sk AS location_date_sk,
    f.sector_sk,
    f.location_sk,
    
    -- MEASURES
    COUNT(CASE WHEN f.active_flag = TRUE THEN 1 END) AS active_jobs,
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) AS new_jobs,
    COUNT(CASE WHEN f.is_soft_delete = TRUE THEN 1 END) AS closed_jobs,
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.company_sk END) AS unique_companies,
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.role_sk END) AS unique_roles,
    
    -- Remote work metrics
    COUNT(CASE WHEN f.active_flag = TRUE AND j.remote_type = 'REMOTE' THEN 1 END) AS remote_jobs,
    COUNT(CASE WHEN f.active_flag = TRUE AND j.remote_type = 'HYBRID' THEN 1 END) AS hybrid_jobs,
    COUNT(CASE WHEN f.active_flag = TRUE AND j.remote_type = 'ONSITE' THEN 1 END) AS onsite_jobs
    
  FROM workspace.warehouse.fact_job_postings f
  LEFT JOIN workspace.warehouse.dim_job j ON f.job_sk = j.job_sk AND j.is_current = TRUE
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
  GROUP BY f.posting_date_sk, f.sector_sk, f.location_sk
),

temporal_metrics AS (
  SELECT
    bm.*,
    
    -- TEMPORAL METRICS: Prior period comparisons
    LAG(bm.active_jobs, 7) OVER (
      PARTITION BY bm.location_sk, bm.sector_sk
      ORDER BY bm.location_date_sk
    ) AS prev_week_active_jobs,
    
    LAG(bm.active_jobs, 30) OVER (
      PARTITION BY bm.location_sk, bm.sector_sk
      ORDER BY bm.location_date_sk
    ) AS prev_month_active_jobs,
    
    -- Month-to-date cumulative new jobs
    SUM(bm.new_jobs) OVER (
      PARTITION BY bm.location_sk, bm.sector_sk,
        YEAR(TO_DATE(CAST(bm.location_date_sk AS STRING), 'yyyyMMdd')),
        MONTH(TO_DATE(CAST(bm.location_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY bm.location_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS mtd_new_jobs,
    
    -- 30-day rolling average
    AVG(bm.new_jobs) OVER (
      PARTITION BY bm.location_sk, bm.sector_sk
      ORDER BY bm.location_date_sk
      ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ) AS rolling_30day_avg_new_jobs
    
  FROM base_metrics bm
)

SELECT
  -- DIMENSIONS
  tm.location_date_sk,
  tm.sector_sk,
  tm.location_sk,
  
  -- MEASURES
  tm.active_jobs,
  tm.new_jobs,
  tm.closed_jobs,
  tm.unique_companies,
  tm.unique_roles,
  
  -- Remote work distribution
  tm.remote_jobs,
  tm.hybrid_jobs,
  tm.onsite_jobs,
  
  -- DERIVED: Remote work ratio
  CASE 
    WHEN tm.active_jobs > 0 
    THEN CAST(tm.remote_jobs AS DECIMAL(10,4)) / tm.active_jobs
    ELSE NULL 
  END AS remote_jobs_ratio,
  
  -- TEMPORAL METRICS
  tm.mtd_new_jobs,
  tm.rolling_30day_avg_new_jobs,
  
  -- DERIVED: Percent changes
  CASE 
    WHEN tm.prev_week_active_jobs > 0 
    THEN CAST((tm.active_jobs - tm.prev_week_active_jobs) AS DECIMAL(10,4)) / tm.prev_week_active_jobs
    ELSE NULL 
  END AS pct_change_vs_prev_week,
  
  CASE 
    WHEN tm.prev_month_active_jobs > 0 
    THEN CAST((tm.active_jobs - tm.prev_month_active_jobs) AS DECIMAL(10,4)) / tm.prev_month_active_jobs
    ELSE NULL 
  END AS pct_change_vs_prev_month,
  
  -- DERIVED: Job density (jobs per company)
  CASE 
    WHEN tm.unique_companies > 0 
    THEN CAST(tm.active_jobs AS DECIMAL(10,2)) / tm.unique_companies
    ELSE NULL 
  END AS jobs_per_company,
  
  -- METADATA
  CURRENT_TIMESTAMP() AS created_at,
  UUID() AS batch_id
  
FROM temporal_metrics tm
ORDER BY tm.location_date_sk DESC, tm.active_jobs DESC;

In [0]:
%sql
-- Validation: Top locations by hiring activity
SELECT
  l.location_name,
  l.city,
  l.state,
  l.country,
  SUM(glt.active_jobs) AS total_active_jobs,
  SUM(glt.new_jobs) AS total_new_jobs,
  COUNT(DISTINCT glt.unique_companies) AS peak_companies,
  ROUND(AVG(glt.remote_jobs_ratio), 4) AS avg_remote_ratio,
  COUNT(DISTINCT glt.location_date_sk) AS days_with_activity
FROM workspace.gold.gold_location_trends glt
JOIN workspace.warehouse.dim_location l ON glt.location_sk = l.location_sk
GROUP BY l.location_name, l.city, l.state, l.country
ORDER BY total_active_jobs DESC
LIMIT 20;